# Day 2 (Evaluation) Practical: Model Evaluation & Interpretability
**AI for Drug Discovery**

In this practical, you will:
1. Implement scaffold splitting for molecular data
2. Compare random vs. scaffold split performance
3. Compute classification and regression metrics
4. Use SHAP to interpret your QSAR model
5. Define an applicability domain
Just as electrophysiology experiments demand proper controls and statistical validation — a patch-clamp recording is meaningless without baseline subtraction, and a calcium imaging experiment requires careful stimulus-vs-spontaneous comparisons — machine learning models in drug discovery require scaffold splitting and rigorous evaluation metrics to ensure predictions generalize beyond memorized training data.

In [ ]:
# Install dependencies: all the Python packages needed for this practical
# rdkit: cheminformatics toolkit for working with molecular structures (SMILES, fingerprints, descriptors)
# scikit-learn: machine learning library providing models (RandomForest) and metrics (RMSE, R2, ROC-AUC)
# xgboost: gradient boosting library, a powerful alternative to RandomForest for QSAR models
# shap: SHapley Additive exPlanations library for model interpretability
# pandas: data manipulation library for tabular data (DataFrames)
# matplotlib: core plotting library for creating figures and visualizations
# seaborn: statistical visualization library built on matplotlib with prettier defaults
# The -q flag suppresses verbose pip output to keep the notebook clean
!pip install rdkit scikit-learn xgboost shap pandas matplotlib seaborn -q

# Import the warnings module from Python's standard library
import warnings
# Suppress all warning messages to keep notebook output clean
# In production code you might want to see these, but for teaching they add noise
warnings.filterwarnings('ignore')

In [ ]:
# Import RDKit's core module for reading/writing molecular structures (SMILES, SDF, etc.)
from rdkit import Chem
# AllChem: extended chemistry functions including fingerprint generation (Morgan/ECFP)
# Descriptors: module for calculating molecular descriptors (MW, LogP, TPSA, etc.)
# Scaffolds: module for Bemis-Murcko scaffold decomposition
from rdkit.Chem import AllChem, Descriptors, Scaffolds
# MurckoScaffold: specifically for extracting the core scaffold (ring systems + linkers) from molecules
from rdkit.Chem.Scaffolds import MurckoScaffold
# rdFingerprintGenerator: new-style API for Morgan (ECFP) fingerprint generation
from rdkit.Chem import rdFingerprintGenerator

# DataStructs: utilities to convert RDKit bit vectors to numpy arrays
from rdkit import DataStructs

# pandas: provides DataFrame for structured data manipulation (like a spreadsheet in Python)
import pandas as pd
# numpy: numerical computing library for efficient array operations and linear algebra
import numpy as np
# matplotlib.pyplot: the primary plotting interface for creating charts, scatter plots, etc.
import matplotlib.pyplot as plt
# seaborn: statistical data visualization, provides prettier defaults and specialized plot types
import seaborn as sns
# RandomForestRegressor: ensemble of decision trees for regression (predicting continuous values like logS)
# RandomForestClassifier: ensemble of decision trees for classification (predicting categories)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
# Import specific evaluation metrics from scikit-learn:
# mean_squared_error: average squared difference between predicted and actual values
# r2_score: coefficient of determination, fraction of variance explained by the model
# roc_auc_score: area under the Receiver Operating Characteristic curve (classification quality)
# precision_recall_curve: trade-off between precision (positive predictive value) and recall (sensitivity)
# roc_curve: trade-off between true positive rate and false positive rate at various thresholds
# average_precision_score: area under the precision-recall curve (summarizes PR trade-off)
# classification_report: text report of precision, recall, F1-score for each class
from sklearn.metrics import (mean_squared_error, r2_score, roc_auc_score,
                             precision_recall_curve, roc_curve, average_precision_score,
                             classification_report)
# train_test_split: utility to randomly split data into training and testing subsets
from sklearn.model_selection import train_test_split
# shap: SHapley Additive exPlanations - a game-theory based approach to explain model predictions
import shap

# Set the default seaborn style to 'whitegrid' for clean, readable plots with grid lines
sns.set_style('whitegrid')
# Confirm all imports loaded successfully (helpful for debugging environment issues)
print('All imports successful!')

## 1. Load Dataset and Generate Features
We'll reuse the Delaney solubility dataset from Day 2.

In [ ]:
# ============================================================
# LOAD THE DELANEY AQUEOUS SOLUBILITY DATASET
# This dataset contains ~1128 small organic molecules with measured
# aqueous solubility (logS) values. Solubility is a critical drug
# property: a drug must dissolve in water to be absorbed into the body.
# Reference: Delaney, J.S. (2004). ESOL. JCICS 44:1000-1009
# ============================================================

# Try to load the dataset from the primary URL (DeepChem's GitHub repository)
# DeepChem is a popular open-source library for deep learning in drug discovery
try:
    # URL pointing to the raw CSV file hosted on DeepChem's GitHub repository
    url = 'https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv'
    # pd.read_csv reads the CSV file from the URL into a pandas DataFrame
    df = pd.read_csv(url)
# If the primary URL fails (network issues, moved file), fall back to an alternative source
except:
    # Alternative URL from Pat Walters' curated collection of cheminformatics datasets
    url = 'https://raw.githubusercontent.com/PatWalters/datafiles/main/delaney.csv'
    # Read the CSV from the backup URL into the same DataFrame variable
    df = pd.read_csv(url)

# Dynamically find the column containing SMILES strings (molecular notation)
# This list comprehension searches all column names for one containing 'smiles' (case-insensitive)
# SMILES = Simplified Molecular-Input Line-Entry System, a text representation of chemical structures
smiles_col = [c for c in df.columns if 'smiles' in c.lower()][0]
# Dynamically find the target column (solubility or logS value)
# We search for columns containing 'solubility' or 'log' to handle different dataset versions
target_col = [c for c in df.columns if 'solubility' in c.lower() or 'log' in c.lower()][0]

# Convert each SMILES string to an RDKit molecule object for cheminformatics operations
# Chem.MolFromSmiles parses the SMILES text and creates a molecular graph object
# Invalid SMILES will return None (handled in the next line)
df['mol'] = df[smiles_col].apply(Chem.MolFromSmiles)
# Remove rows where molecule parsing failed (mol is None/NaN)
# .notna() creates a boolean mask, we filter to keep only valid molecules
# .reset_index(drop=True) re-numbers the rows from 0 to avoid gaps in the index
df = df[df['mol'].notna()].reset_index(drop=True)
# Print how many valid molecules remain after filtering
print(f'Loaded {len(df)} valid molecules')

In [ ]:
# ============================================================
# GENERATE MOLECULAR FEATURES: FINGERPRINTS + DESCRIPTORS
# We create two types of features:
# 1. Morgan fingerprints (ECFP): binary vectors encoding molecular substructures
# 2. Molecular descriptors: numerical properties like molecular weight, LogP, etc.
# These features are the input (X) to our machine learning model.
# ============================================================

# Function to convert an RDKit molecule object to a Morgan (ECFP) fingerprint bit vector
# Morgan fingerprints encode circular substructures around each atom
# radius=2 means we consider substructures up to 2 bonds from each atom (equivalent to ECFP4)
# n_bits=2048 is the length of the fingerprint vector (2048-bit hashed fingerprint)
def mol_to_fp(mol, radius=2, n_bits=2048):
    # Generate Morgan fingerprint as an RDKit bit vector object (new generator API)
    # radius=2 captures local chemical environments (functional groups, ring systems)
    # nBits=2048 hashes the substructure identifiers into a fixed-length binary vector
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits).GetFingerprint(mol)
    # Create a zero-filled numpy array to hold the fingerprint bits
    # dtype=np.int8 uses minimal memory (each bit stored as 0 or 1)
    arr = np.zeros(n_bits, dtype=np.int8)
    # Convert the RDKit fingerprint object into the numpy array in-place
    # This bridges RDKit's internal format with numpy for use in scikit-learn
    DataStructs.ConvertToNumpyArray(fp, arr)
    # Return the numpy array representation of the fingerprint
    return arr

# Function to calculate 10 common molecular descriptors for a molecule
# These descriptors capture global physicochemical properties relevant to drug-likeness
def calc_descriptors(mol):
    # Return a list of 10 descriptors:
    # MolWt: Molecular weight in Daltons (larger molecules tend to be less soluble)
    # MolLogP: Wildman-Crippen LogP (octanol-water partition coefficient, measures hydrophobicity)
    # NumHDonors: Number of hydrogen bond donors (OH, NH groups)
    # NumHAcceptors: Number of hydrogen bond acceptors (O, N atoms with lone pairs)
    # TPSA: Topological Polar Surface Area in Angstrom^2 (correlates with membrane permeability)
    # NumRotatableBonds: Number of freely rotatable bonds (molecular flexibility)
    # NumAromaticRings: Count of aromatic ring systems (planar, conjugated rings)
    # HeavyAtomCount: Number of non-hydrogen atoms (molecular size indicator)
    # RingCount: Total number of rings (complexity measure)
    # FractionCSP3: Fraction of sp3-hybridized carbons (3D shape complexity, "flatness")
    return [Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
            Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
            Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol),
            Descriptors.NumAromaticRings(mol), Descriptors.HeavyAtomCount(mol),
            Descriptors.RingCount(mol), Descriptors.FractionCSP3(mol)]

# Human-readable names for the 10 descriptors, used for SHAP plots and feature importance
desc_names = ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds', 'AromaticRings',
              'HeavyAtoms', 'RingCount', 'FractionCSP3']

# Generate Morgan fingerprints for ALL molecules in the dataset
# List comprehension applies mol_to_fp to each molecule, np.array stacks into a 2D matrix
fp_array = np.array([mol_to_fp(m) for m in df['mol']])
# Generate molecular descriptors for ALL molecules in the dataset
# Same pattern: list comprehension + np.array creates a (n_molecules, 10) matrix
desc_array = np.array([calc_descriptors(m) for m in df['mol']])
# Concatenate fingerprints and descriptors horizontally into a single feature matrix
# Result: each row is one molecule, columns are [2048 FP bits | 10 descriptors] = 2058 features
X = np.hstack([fp_array, desc_array])
# Extract the target variable (measured logS solubility values) as a numpy array
y = df[target_col].values

# Create a list of feature names matching the columns of X
# First 2048 names are FP_0 through FP_2047, then the 10 descriptor names
feature_names = [f'FP_{i}' for i in range(2048)] + desc_names
# Print the dimensions to confirm: (n_molecules, 2058) for X and (n_molecules,) for y
print(f'Feature matrix: {X.shape}, Target: {y.shape}')

## 2. Scaffold Splitting

### What is Bemis-Murcko Scaffold Decomposition?

The **Bemis-Murcko decomposition** (Bemis & Murcko, 1996) is a method for extracting the **core molecular framework** from a drug-like molecule. It works by:

1. **Identifying all ring systems** in the molecule (aromatic and aliphatic rings)
2. **Keeping the linker chains** that connect ring systems to each other
3. **Removing all side-chain substituents** (R-groups) that branch off from the framework

For example, if you have a molecule with a benzene ring connected to a pyridine ring via an ethyl linker, with various substituents attached, the Bemis-Murcko scaffold would be just `c1ccc(CCc2ccccn2)cc1` — the two rings plus the linker, with all substituents stripped away.

This decomposition groups molecules into **chemical series**: molecules that share the same scaffold are structural analogs of each other. They were likely designed by the same medicinal chemistry team, optimizing the same lead compound.

### Why Are Random Splits Dangerous in Drug Discovery?

In a **random train/test split**, molecules from the same chemical series can end up in both the training and test sets. This is a form of **data leakage**: the model "sees" close analogs during training, making test-set prediction trivially easy. The resulting metrics are **optimistically biased** — they make the model look better than it actually is.

In real-world drug discovery, when you deploy a model, you want to predict activity for **novel chemical series** that the model has never seen. Random splits do not simulate this scenario at all.

**Scaffold splitting** ensures that all molecules with the same Murcko scaffold stay together in either the training OR the test set — never both. This provides a much more realistic estimate of how the model will perform on **genuinely new** chemical matter.

Studies have shown that performance drops of 10-30% are common when switching from random to scaffold splits (Wallach & Heifets, 2018), revealing that many published QSAR models were overfitting to structural memorization rather than learning true structure-activity relationships.

**Reference:** Bemis, G.W. & Murcko, M.A. (1996). The Properties of Known Drugs. J. Med. Chem. 39:2887-2893

### The Neuroscience Parallel: Why Controls Matter Everywhere

The principle behind scaffold splitting — preventing data leakage through structural similarity — has a direct parallel in neuroscience experimental design. In electrophysiology experiments like those conducted in Dr. Serbe-Kamp's research on *Drosophila* visual motion circuits, **proper controls are non-negotiable**. A stimulus-evoked response recorded via patch-clamp or calcium imaging is only meaningful when compared against **spontaneous baseline activity**. Pharmacological experiments (e.g., applying a GABA-A agonist to dissected fly brains) require rigorous **wash-in/wash-out controls** to confirm that any observed change in neural firing is truly drug-induced and not an artifact of preparation drift, photobleaching, or mechanical disturbance.

Similarly, in QSAR modeling, scaffold splitting serves as the "proper control" that prevents the confound of structural memorization. A random split is analogous to a neuroscience experiment without a baseline condition — the results may look impressive, but they cannot be trusted. Just as a well-designed electrophysiology experiment separates stimulus-driven signals from noise, scaffold splitting separates genuine structure–activity learning from trivial pattern matching on shared scaffolds.

This is especially critical for **CNS (central nervous system) drug targets**. Many neuroactive compounds targeting **GABA-A receptors** share the benzodiazepine scaffold (e.g., diazepam, alprazolam, midazolam), while compounds targeting **serotonin receptors** often share the SSRI (selective serotonin reuptake inhibitor) scaffold (e.g., fluoxetine, sertraline). If a QSAR model for GABA-A modulators is evaluated with a random split, molecules from the same benzodiazepine series will appear in both train and test sets, giving an inflated R² that collapses when the model encounters a structurally novel anxiolytic. Scaffold splitting exposes this fragility and drives development of models that truly understand the pharmacophore rather than memorizing chemical series.

In [ ]:
# ============================================================
# EXTRACT BEMIS-MURCKO SCAFFOLDS FROM EACH MOLECULE
# The scaffold is the core ring system plus linker chains of a molecule,
# with all side chains removed. Molecules sharing the same scaffold
# belong to the same chemical series and are structurally similar.
# ============================================================

# Function to extract the Bemis-Murcko scaffold from an RDKit molecule object
# and return it as a canonical SMILES string
def get_scaffold(mol):
    # Try-except to handle any rare molecules where scaffold extraction fails
    try:
        # GetScaffoldForMol extracts the Murcko framework:
        # keeps all ring systems and the linker chains connecting them,
        # removes all side chain substituents (R-groups)
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        # Convert the scaffold molecule back to a canonical SMILES string
        # Canonical SMILES ensures the same scaffold always produces the same string
        return Chem.MolToSmiles(scaffold)
    except:
        # If scaffold extraction fails, return empty string (will be grouped together)
        return ''

# Apply the get_scaffold function to every molecule in the DataFrame
# This creates a new column 'scaffold' with the Murcko scaffold SMILES for each molecule
df['scaffold'] = df['mol'].apply(get_scaffold)
# Print the number of unique scaffolds vs. total molecules
# A high ratio means diverse chemical series; a low ratio means many analogs
print(f'Unique scaffolds: {df["scaffold"].nunique()} / {len(df)} molecules')
# Show the most common scaffolds and how many molecules share each one
print(f'Top 5 scaffolds by frequency:')
# value_counts() counts occurrences of each scaffold, .head() shows the top 5
print(df['scaffold'].value_counts().head())

In [ ]:
# ============================================================
# SCAFFOLD-BASED TRAIN/TEST SPLIT
# Unlike random splitting, scaffold splitting ensures that NO chemical
# scaffold appears in BOTH training and test sets. This prevents data
# leakage from structurally similar molecules and gives a more realistic
# estimate of how the model will perform on truly novel chemical series.
# ============================================================

# Function to perform scaffold-based splitting of the dataset
# df: the DataFrame containing molecules and their scaffold assignments
# test_frac: fraction of data to use for testing (0.2 = 20%)
# random_state: seed for reproducibility of the random scaffold ordering
def scaffold_split(df, test_frac=0.2, random_state=42):
    # Extract the scaffold labels as a numpy array for fast indexing
    scaffolds = df['scaffold'].values
    # Get a list of all unique scaffolds (each represents a chemical series)
    unique_scaffolds = list(set(scaffolds))
    # Set the random seed for reproducibility (same seed = same split every time)
    np.random.seed(random_state)
    # Randomly shuffle the order of scaffolds (determines which go to test set)
    np.random.shuffle(unique_scaffolds)

    # Calculate how many molecules should be in the test set
    test_size = int(len(df) * test_frac)
    # Initialize an empty list to accumulate test set molecule indices
    test_indices = []
    # Iterate through shuffled scaffolds, adding entire scaffold groups to test set
    # This ensures all molecules with the same scaffold stay together (no leakage!)
    for scaffold in unique_scaffolds:
        # Find all molecule indices that have this scaffold
        indices = np.where(scaffolds == scaffold)[0].tolist()
        # Add all molecules from this scaffold to the test set
        test_indices.extend(indices)
        # Stop once we have enough test molecules (at least test_frac of the data)
        if len(test_indices) >= test_size:
            break

    # Convert test indices to a set for O(1) lookup performance
    test_idx = set(test_indices)
    # Training set = all indices NOT in the test set
    train_idx = [i for i in range(len(df)) if i not in test_idx]
    # Return train and test indices as lists
    return train_idx, list(test_idx)

# Execute the scaffold split and store the resulting indices
train_idx_scaffold, test_idx_scaffold = scaffold_split(df)
# Print the sizes of training and test sets to confirm the split worked correctly
print(f'Scaffold split - Train: {len(train_idx_scaffold)}, Test: {len(test_idx_scaffold)}')

## 3. Compare Random vs. Scaffold Split

### Evaluation Metrics for Regression

When comparing the two splitting strategies, we use two key regression metrics:

**RMSE (Root Mean Squared Error):**
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

RMSE measures the average magnitude of prediction errors, in the same units as the target variable (logS). It penalizes large errors more heavily than small ones due to the squaring operation. **Lower is better.** For solubility prediction, an RMSE of ~0.7 logS units is considered good.

**MAE (Mean Absolute Error):**
$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

MAE is the average of absolute errors. Unlike RMSE, it weights all errors equally. It's more robust to outliers. **Lower is better.**

**R² (Coefficient of Determination):**
$$R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$$

R² measures what fraction of the target's variance is explained by the model. R²=1.0 means perfect predictions; R²=0.0 means the model is no better than predicting the mean; R²<0 means the model is worse than the mean. **Higher is better.**

**For classification tasks** (e.g., active/inactive), we would use:

**ROC-AUC (Area Under the Receiver Operating Characteristic Curve):**
The ROC curve plots True Positive Rate (sensitivity) vs. False Positive Rate (1 - specificity) at every classification threshold. AUC=1.0 means perfect separation; AUC=0.5 means random guessing. ROC-AUC is threshold-independent and measures overall ranking quality.

**Precision-Recall AUC (PR-AUC / Average Precision):**
Precision = TP/(TP+FP) (of all positive predictions, how many are correct?). Recall = TP/(TP+FN) (of all actual positives, how many were found?). PR-AUC is especially important when classes are **imbalanced** (e.g., few active molecules among many inactives), because unlike ROC-AUC, it is sensitive to the ratio of positives to negatives.

### What to Expect

You should observe that **random split gives better-looking metrics** (lower RMSE, higher R²) than scaffold split. This is NOT because the model is actually better — it's because the random test set contains molecules that are structurally similar to training molecules, making prediction easier (data leakage).

In [ ]:
# ============================================================
# COMPARE RANDOM VS. SCAFFOLD SPLIT PERFORMANCE
# This comparison demonstrates why the splitting strategy matters:
# random splits leak structural information, scaffold splits don't.
# ============================================================

# --- RANDOM SPLIT ---
# train_test_split randomly assigns 80% of molecules to training and 20% to testing
# This is the naive approach: structurally similar molecules can appear in both sets
# test_size=0.2 means 20% goes to test; random_state=42 ensures reproducibility
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=42)

# --- SCAFFOLD SPLIT ---
# Use the pre-computed scaffold-based indices to create training features/targets
# Training set: molecules whose scaffolds are NOT in the test set
X_train_scaf = X[train_idx_scaffold]
# Test set: molecules whose scaffolds are unique to the test set
X_test_scaf = X[test_idx_scaffold]
# Training target values (measured logS) for scaffold-split training molecules
y_train_scaf = y[train_idx_scaffold]
# Test target values for scaffold-split test molecules
y_test_scaf = y[test_idx_scaffold]

# --- TRAIN RANDOM FOREST ON RANDOM SPLIT ---
# RandomForestRegressor: ensemble of 500 decision trees that average their predictions
# n_estimators=500: number of trees (more trees = more stable but slower)
# random_state=42: reproducibility; n_jobs=-1: use all available CPU cores for speed
rf_rand = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
# Fit (train) the model on the randomly-split training data
rf_rand.fit(X_train_rand, y_train_rand)
# Generate predictions for the random-split test set
y_pred_rand = rf_rand.predict(X_test_rand)

# --- TRAIN RANDOM FOREST ON SCAFFOLD SPLIT ---
# Same model architecture, but trained on scaffold-split data (no structural leakage)
rf_scaf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
# Fit the model on the scaffold-split training data
rf_scaf.fit(X_train_scaf, y_train_scaf)
# Generate predictions for the scaffold-split test set
y_pred_scaf = rf_scaf.predict(X_test_scaf)

# --- PRINT COMPARISON OF METRICS ---
# Random split results (expected to look "better" due to data leakage)
print('Random Split:')
# RMSE = Root Mean Squared Error: sqrt(mean((actual - predicted)^2))
# Lower is better. Penalizes large errors more than MAE.
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test_rand, y_pred_rand)):.3f}')
# R2 = Coefficient of determination: 1 - SS_res/SS_tot
# 1.0 = perfect predictions, 0.0 = predicting the mean, negative = worse than mean
print(f'  R2:   {r2_score(y_test_rand, y_pred_rand):.3f}')
print()
# Scaffold split results (expected to look "worse" but are more realistic!)
print('Scaffold Split:')
# RMSE for scaffold split - typically higher because test molecules are truly novel
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test_scaf, y_pred_scaf)):.3f}')
# R2 for scaffold split - typically lower, reflecting real-world performance
print(f'  R2:   {r2_score(y_test_scaf, y_pred_scaf):.3f}')
print()
# Key takeaway: random splits are optimistically biased because structurally similar
# molecules appear in both train and test, making prediction artificially easy
print('Notice how scaffold split gives lower (more realistic) performance!')

In [ ]:
# ============================================================
# VISUALIZE PREDICTED VS. ACTUAL VALUES FOR BOTH SPLIT STRATEGIES
# A perfect model would have all points on the diagonal (y=x) line.
# Scatter around the line indicates prediction error.
# ============================================================

# Create a figure with two side-by-side subplots (1 row, 2 columns)
# figsize=(14, 6) makes the figure 14 inches wide and 6 inches tall
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Loop over both splits, plotting actual vs. predicted for each
# zip pairs up: axes[0] with random data, axes[1] with scaffold data
for ax, yt, yp, title in zip(axes,
    [y_test_rand, y_test_scaf], [y_pred_rand, y_pred_scaf],
    ['Random Split', 'Scaffold Split']):
    # Scatter plot: each point is one test molecule (actual logS vs. predicted logS)
    # alpha=0.5 makes points semi-transparent to show density/overlap
    # s=30 sets the marker size
    ax.scatter(yt, yp, alpha=0.5, s=30)
    # Plot the ideal y=x diagonal line in red dashes
    # Points on this line represent perfect predictions (predicted == actual)
    ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
    # Calculate RMSE for this split to display in the title
    rmse = np.sqrt(mean_squared_error(yt, yp))
    # Calculate R-squared for this split to display in the title
    r2 = r2_score(yt, yp)
    # Set the subplot title showing split type and both metrics
    ax.set_title(f'{title}\nRMSE={rmse:.3f}, R2={r2:.3f}', fontsize=13)
    # Label the x-axis as the actual (measured) solubility value
    ax.set_xlabel('Actual logS')
    # Label the y-axis as the model's predicted solubility value
    ax.set_ylabel('Predicted logS')
    # Equal aspect ratio ensures the diagonal line is at 45 degrees
    ax.set_aspect('equal')

# Adjust subplot spacing to prevent label overlap
plt.tight_layout()
# Render and display the figure
plt.show()

## 4. SHAP Analysis

### Shapley Values: From Game Theory to Machine Learning

**SHAP (SHapley Additive exPlanations)** is based on **Shapley values** from cooperative game theory, introduced by Lloyd Shapley in 1953 (for which he received the Nobel Prize in Economics in 2012).

**The Game Theory Analogy:**
Imagine a team of players (features) working together to achieve a "payout" (the model's prediction). The Shapley value for each player is the **fair allocation** of the total payout, considering all possible coalitions (subsets) of players. Formally, the Shapley value for feature $j$ is:

$$\phi_j = \sum_{S \subseteq N \setminus \{j\}} \frac{|S|!(|N|-|S|-1)!}{|N|!} \left[ f(S \cup \{j\}) - f(S) \right]$$

where $N$ is the set of all features, $S$ is a subset not containing feature $j$, and $f(S)$ is the model's prediction using only features in $S$. This computes the **marginal contribution** of feature $j$ averaged over all possible orderings of features.

**Key properties of Shapley values:**
1. **Efficiency**: SHAP values sum to the difference between the prediction and the base value (average prediction)
2. **Symmetry**: Features that contribute equally get equal SHAP values
3. **Dummy**: Features that don't contribute get SHAP value = 0
4. **Additivity**: SHAP values for combined models are the sum of individual SHAP values

### How TreeSHAP Works

Computing exact Shapley values requires evaluating $2^n$ subsets for $n$ features — intractable for 2058 features! **TreeSHAP** (Lundberg et al., 2020) exploits the structure of decision trees to compute exact SHAP values in **polynomial time** $O(TLD^2)$ where $T$ = number of trees, $L$ = maximum leaves, $D$ = maximum depth.

TreeSHAP works by recursively tracking how each feature's inclusion/exclusion affects the prediction path through each tree. For features not used in a tree's decision path, their SHAP value is zero for that tree.

### Interpreting SHAP Plots

- **Beeswarm plot**: Each dot is one molecule. X-axis = SHAP value (impact on prediction). Color = feature value (red = high, blue = low). A feature where red dots are on the right means high values of that feature increase the prediction.
- **Waterfall plot**: Decomposes a single prediction. Shows how each feature pushes the prediction up or down from the base value (average). Red bars push prediction higher, blue bars push lower.

For solubility prediction, we expect LogP to have negative SHAP values (higher LogP → lower solubility) and TPSA to have positive values (higher polar surface area → better water solubility).

**Reference:** Lundberg, S.M. & Lee, S. (2017). A Unified Approach to Interpreting Model Predictions. NeurIPS.

### Neuroscience Application: Interpreting Predictions on Neural Targets

SHAP analysis is particularly powerful when applied to **neuroscience drug targets**, because it can reveal which molecular features drive predicted activity in ways that align with known pharmacology. For example, when building a QSAR model for **GABA-A receptor modulators**, SHAP might highlight the importance of **aromatic nitrogen atoms** and **fused ring systems** — features characteristic of the benzodiazepine pharmacophore — as well as **chlorine substituents** at specific positions (as seen in chlordiazepoxide, the first marketed benzodiazepine). If the model instead highlights irrelevant features (e.g., aliphatic chain length), this signals potential overfitting or data artifacts.

This interpretability parallels how neuroscientists decompose complex circuit-level phenomena into individual neuronal contributions. In Dr. Serbe-Kamp's work on **T4/T5 direction-selective neurons** in the *Drosophila* visual system, calcium imaging is used to identify **which specific neurons in a circuit contribute most to a behavioral output** (such as the optomotor response to visual motion). Just as SHAP assigns each molecular feature a quantitative contribution to a predicted bioactivity, calcium imaging with genetically encoded indicators (e.g., GCaMP) assigns each neuron a quantitative contribution to a population-level signal. In both cases, the goal is the same: moving from a "black box" (an opaque model prediction or an unexplained behavior) to a mechanistic understanding of **which components matter and why**.

In [ ]:
# ============================================================
# SHAP (SHapley Additive exPlanations) ANALYSIS
# SHAP values explain how much each feature contributes to each prediction.
# Based on Shapley values from cooperative game theory (Lloyd Shapley, 1953).
# TreeSHAP is an efficient algorithm specifically for tree-based models.
# ============================================================

# Use a subset of 100 test molecules for SHAP analysis (full set would be slow)
# SHAP computation scales with n_samples * n_features * n_trees
X_explain = X_test_scaf[:100]

# Create a TreeExplainer for the scaffold-split Random Forest model
# TreeExplainer uses the exact tree structure to compute SHAP values in polynomial time
# (much faster than the model-agnostic KernelSHAP which uses sampling)
explainer = shap.TreeExplainer(rf_scaf)
# Compute SHAP values for each feature for each of the 100 test molecules
# Result shape: (100 molecules, 2058 features) - one SHAP value per feature per molecule
# Positive SHAP value = feature pushes prediction higher; negative = pushes prediction lower
shap_values = explainer.shap_values(X_explain)

# Print the shape to confirm: should be (100, 2058) = 100 molecules x 2058 features
print(f'SHAP values shape: {shap_values.shape}')
# Confirm SHAP analysis completed without errors
print('SHAP analysis complete!')

In [ ]:
# ============================================================
# SHAP BEESWARM SUMMARY PLOT FOR MOLECULAR DESCRIPTORS
# This plot shows the distribution of SHAP values for each feature.
# Each dot is one molecule. Color = feature value. Position = SHAP value.
# Features are ordered by overall importance (mean |SHAP|).
# ============================================================

# Extract SHAP values for just the 10 descriptor features (last 10 columns)
# We skip the 2048 fingerprint bits because showing 2048 features would be unreadable
desc_shap = shap_values[:, -len(desc_names):]
# Extract the actual descriptor feature values for the same 100 test molecules
# These values determine the dot colors in the beeswarm plot
desc_X = X_explain[:, -len(desc_names):]

# Create a new figure with specified dimensions
plt.figure(figsize=(10, 6))
# Generate the SHAP beeswarm summary plot
# Each row = one feature; each dot = one molecule
# x-axis = SHAP value (impact on prediction); color = feature value (red=high, blue=low)
# show=False prevents shap from calling plt.show() so we can add a title first
shap.summary_plot(desc_shap, desc_X, feature_names=desc_names, show=False)
# Add a descriptive title to the plot
plt.title('SHAP Feature Importance (Descriptors)', fontsize=14)
# Adjust layout to prevent label clipping
plt.tight_layout()
# Render and display the plot
plt.show()

In [ ]:
# ============================================================
# SHAP WATERFALL PLOT FOR A SINGLE MOLECULE
# This plot shows how each feature moves the prediction from the
# base value (average prediction) to the final prediction for ONE molecule.
# It decomposes a single prediction into additive feature contributions.
# ============================================================

# Select the first test molecule (index 0) to explain
# You can change this to any index from 0 to 99 to explain a different molecule
idx = 0  # First test molecule
# Create a new figure for the waterfall plot
plt.figure(figsize=(10, 6))
# Generate a SHAP waterfall plot using the Explanation object
# values: SHAP values for molecule idx (how much each feature contributed)
# base_values: the expected (average) model output across the training set
# data: the actual feature values for this molecule (shown next to feature names)
# feature_names: human-readable names for each feature
shap.waterfall_plot(shap.Explanation(
    values=desc_shap[idx],
    # expected_value may be a 1-element array in newer SHAP versions — extract scalar
    base_values=float(np.ravel(explainer.expected_value)[0]),
    data=desc_X[idx],
    feature_names=desc_names
), show=False)
# Add a title identifying which molecule is being explained
plt.title(f'SHAP Explanation for Molecule {idx}', fontsize=14)
# Adjust layout to prevent label clipping
plt.tight_layout()
# Render and display the plot
plt.show()

## 5. Applicability Domain

### What is the Applicability Domain?

The **Applicability Domain (AD)** defines the region of chemical space where a QSAR model's predictions are expected to be reliable. Predictions for molecules **outside** the AD should be treated with skepticism — the model is extrapolating beyond its training experience.

This is a critical concept in drug discovery: a model trained on kinase inhibitors should NOT be trusted to predict solubility of peptides or polymers, even if it reports a confident-looking number.

### Methods for Defining the Applicability Domain

**1. Distance-Based Methods (used in this practical):**
Measure the **Tanimoto similarity** between a query molecule and its nearest neighbor in the training set. If the maximum similarity is below a threshold (e.g., 0.4), the molecule is considered outside the AD. The Tanimoto coefficient for two binary fingerprints $A$ and $B$ is:

$$T(A,B) = \frac{|A \cap B|}{|A \cup B|} = \frac{c}{a + b - c}$$

where $a = |A|$, $b = |B|$, and $c$ = number of bits set in both. This ranges from 0 (no common bits) to 1 (identical fingerprints).

**2. Descriptor-Range Methods (Bounding Box):**
Define the AD as the hypercube bounded by the min/max values of each descriptor in the training set. A molecule is outside the AD if ANY descriptor falls outside its training range. This is simple but can be too permissive (the actual training data may not fill the entire hypercube).

**3. Leverage-Based Methods (Williams Plot):**
Calculate the **leverage** $h_i$ of each molecule:
$$h_i = x_i^T (X^T X)^{-1} x_i$$
where $x_i$ is the feature vector and $X$ is the training matrix. High leverage indicates the molecule is far from the center of the training distribution.

**4. Conformal Prediction:**
A statistically rigorous framework that provides **prediction intervals** with guaranteed coverage probability. For a significance level $\epsilon$, conformal prediction guarantees that the true value falls within the prediction interval at least $(1-\epsilon)$ of the time, regardless of the underlying distribution. This is more principled than ad-hoc thresholds.

### Why the AD Matters for Drug Discovery

In a real drug discovery project, you might screen millions of virtual molecules. The AD helps you:
- **Flag unreliable predictions** before wasting resources on synthesis and testing
- **Prioritize** molecules that are within the model's reliable prediction space
- **Guide** medicinal chemists toward chemical space where the model is trustworthy

**Reference:** Sahigara, F. et al. (2012). Comparison of Different Approaches to Define the Applicability Domain. Molecules 17:4791-4810

In [ ]:
# ============================================================
# APPLICABILITY DOMAIN (AD) ANALYSIS
# The applicability domain defines the region of chemical space
# where the model's predictions are reliable. Molecules too different
# from the training set (outside the AD) may get unreliable predictions.
# We use Tanimoto similarity to measure "closeness" to training data.
# ============================================================

# Import DataStructs for Tanimoto similarity calculation between fingerprints
from rdkit import DataStructs

# Pre-compute Morgan fingerprints for ALL training molecules
# These will be compared against each test molecule to find the most similar training neighbor
# We use the same fingerprint settings (radius=2, 2048 bits) as our features
train_fps = [rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(df['mol'].iloc[i])
             for i in train_idx_scaffold]

# Function to find the maximum Tanimoto similarity between a test molecule and all training molecules
# Tanimoto similarity = |A ∩ B| / |A ∪ B| for two bit sets A and B
# Range: 0 (completely different) to 1 (identical fingerprints)
def max_tanimoto_to_train(mol, train_fps):
    # Generate the Morgan fingerprint for the test molecule
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol)
    # Compute Tanimoto similarity between this molecule and EVERY training molecule
    sims = [DataStructs.TanimotoSimilarity(fp, tfp) for tfp in train_fps]
    # Return the highest similarity (nearest neighbor in fingerprint space)
    return max(sims)

# Get the RDKit molecule objects for all test molecules using scaffold split indices
test_mols = [df['mol'].iloc[i] for i in test_idx_scaffold]
# Compute the maximum Tanimoto similarity to the training set for the first 100 test molecules
# This tells us how "familiar" each test molecule is to the model
max_sims = [max_tanimoto_to_train(m, train_fps) for m in test_mols[:100]]

# Calculate the absolute prediction error for the first 100 test molecules
# |actual - predicted| gives us the magnitude of error regardless of direction
errors = np.abs(y_test_scaf[:100] - y_pred_scaf[:100])

# Create a scatter plot showing the relationship between similarity and error
plt.figure(figsize=(10, 6))
# Each point is one test molecule: x = similarity to training set, y = prediction error
# We expect molecules more similar to training (higher x) to have lower error (lower y)
plt.scatter(max_sims, errors, alpha=0.5, s=40)
# Label axes clearly for interpretation
plt.xlabel('Max Tanimoto Similarity to Training Set', fontsize=12)
plt.ylabel('Absolute Prediction Error (logS)', fontsize=12)
# Add a descriptive title
plt.title('Applicability Domain: Similarity vs. Error', fontsize=14)
# Draw a vertical red dashed line at similarity = 0.4 as the AD threshold
# Molecules left of this line (< 0.4 similarity) are considered outside the AD
plt.axvline(0.4, color='red', linestyle='--', label='AD threshold = 0.4')
# Add a legend to identify the threshold line
plt.legend()
# Adjust layout to prevent label clipping
plt.tight_layout()
# Render and display the plot
plt.show()

# --- COMPUTE AND PRINT AD STATISTICS ---
# Create a boolean mask: True for molecules inside the AD (similarity >= 0.4)
in_ad = np.array(max_sims) >= 0.4
# Print how many molecules fall inside the AD
print(f'Molecules in AD (similarity >= 0.4): {in_ad.sum()} / {len(max_sims)}')
# Calculate and print RMSE only for molecules INSIDE the AD (should be lower/better)
print(f'RMSE in AD: {np.sqrt(np.mean(errors[in_ad]**2)):.3f}')
# If there are molecules outside the AD, calculate their RMSE too (should be higher/worse)
if (~in_ad).sum() > 0:
    print(f'RMSE outside AD: {np.sqrt(np.mean(errors[~in_ad]**2)):.3f}')

## Human Physiology: ECG, EMG & Drug Safety Evaluation

### Bridging Computational Prediction with Physiological Measurement

The evaluation principles we have studied in this practical — careful train/test splitting, rigorous metrics, model interpretability, and applicability domains — apply not only to computational QSAR models but also to **physiological experiments** used in drug safety evaluation. In fact, some of the most important drug safety assays involve measuring electrical signals from the human body.

### The Electrocardiogram (ECG) and Cardiac Safety

The **electrocardiogram (ECG or EKG)** records the electrical activity of the heart over time using electrodes placed on the skin. A single heartbeat produces a characteristic waveform with distinct components:

- **P wave**: Represents **atrial depolarization** — the electrical signal that triggers contraction of the atria (upper heart chambers). Duration: ~80-100 ms.
- **QRS complex**: Represents **ventricular depolarization** — the powerful electrical signal that triggers contraction of the ventricles (lower heart chambers, which pump blood to the body). The R wave is the tallest peak on the ECG. Duration: ~80-120 ms.
- **T wave**: Represents **ventricular repolarization** — the electrical "reset" of the ventricles as ion channels restore the resting membrane potential. This phase depends critically on potassium (K⁺) channels.
- **QT interval**: The time from the beginning of the Q wave to the end of the T wave. This represents the **total duration of ventricular electrical activity** (depolarization + repolarization). Normal QTc (corrected for heart rate): 350-450 ms.

### QT Prolongation: The #1 Cardiac Safety Concern in Drug Development

**QT prolongation** — an abnormally long QT interval — is the **single most important cardiac safety concern** in pharmaceutical development. It indicates that the ventricles are taking too long to repolarize, which creates a vulnerability window for dangerous re-entrant arrhythmias, particularly **Torsades de Pointes (TdP)**, a polymorphic ventricular tachycardia that can degenerate into ventricular fibrillation and sudden cardiac death.

**The hERG Channel (Kv11.1):**
The primary molecular target responsible for drug-induced QT prolongation is the **hERG potassium channel** (human Ether-à-go-go-Related Gene, also known as Kv11.1). This voltage-gated K⁺ channel conducts the rapid delayed rectifier potassium current (IKr), which is essential for cardiac repolarization. The hERG channel has an unusually large and hydrophobic inner cavity that makes it promiscuously bind many drug molecules — a molecular "sticky trap" for drugs.

When a drug blocks the hERG channel:
1. K⁺ efflux during repolarization is reduced
2. Repolarization takes longer → QT interval extends
3. The prolonged vulnerable period allows early afterdepolarizations (EADs)
4. EADs can trigger Torsades de Pointes → ventricular fibrillation → sudden death

**Drugs Withdrawn from Market Due to QT Prolongation:**
- **Terfenadine** (Seldane, antihistamine) — withdrawn 1998; caused fatal arrhythmias when combined with CYP3A4 inhibitors
- **Cisapride** (Propulsid, GI motility agent) — withdrawn 2000; caused QT prolongation and >80 reported deaths
- **Astemizole** (Hismanal, antihistamine) — withdrawn 1999; potent hERG blocker discovered post-marketing

These high-profile withdrawals led to the **ICH S7B guideline**, which now **mandates hERG channel testing** for ALL drug candidates before they can enter clinical trials. This includes:
- In vitro hERG patch-clamp electrophysiology
- In vivo QT studies in animals (typically dogs or non-human primates)
- Thorough QT/QTc (TQT) studies in human volunteers (ICH E14)

### EMG (Electromyography) and Neuromuscular Safety

**Electromyography (EMG)** records electrical signals from skeletal muscles. It is used to assess:
- **Motor unit recruitment**: As muscle force increases, more motor units (a motor neuron + all muscle fibers it innervates) are recruited, and firing rates increase. This is the **size principle** (Henneman, 1957).
- **Muscle fatigue analysis**: During sustained contraction, EMG signals show characteristic changes: decreased median frequency and increased amplitude as larger, fatigable motor units are recruited.
- **Drug-induced myopathy**: Some drugs (e.g., statins, corticosteroids) can cause muscle damage detectable by EMG changes.

### Hands-On SpikerBot Experiments Students Can Try

Students with access to **Backyard Brains SpikerBox** or similar bioamplifiers can:
1. **Record your own ECG**: Place electrodes on wrists and ankle, measure the QT interval from your R-R interval
2. **Measure QT interval**: Use Bazett's correction formula: QTc = QT / √(RR interval in seconds)
3. **Record EMG from forearm**: Clench your fist at different force levels, observe motor unit recruitment
4. **Fatigue experiment**: Sustain a grip and watch the EMG signal change over 60 seconds

### The Evaluation Parallel: QSAR ↔ Physiology

| QSAR Model Evaluation | Physiological Experiment |
|---|---|
| Training set | Baseline (pre-drug) measurements |
| Test set | Post-drug measurements |
| Scaffold split (avoiding structural leakage) | Crossover design (same subject, washout period) |
| RMSE, R² metrics | QT prolongation magnitude (ms), EMG amplitude |
| SHAP feature importance | Which ECG features change most with the drug? |
| Applicability domain | Patient population where drug effect is characterized |
| Conformal prediction intervals | Confidence intervals on QTc change (ΔΔQTc) |

The **same rigorous evaluation mindset** applies whether you are validating a computational model or a physiological measurement. Always ask: Is my comparison fair? Am I measuring what I think I'm measuring? How confident am I in the result?

**Reference:** Redfern, W.S. et al. (2003). Relationships between preclinical cardiac electrophysiology, clinical QT interval prolongation and torsade de pointes for a broad range of drugs. *Cardiovasc. Res.* 58:32-45

### From hERG to GluCl: Ion Channel Safety Across Organ Systems

While the hERG potassium channel is the primary **cardiac** safety target, the same class of ion channel biology governs safety in the **nervous system**. **GluCl (glutamate-gated chloride channels)**, studied extensively in invertebrate neuroscience, are the molecular targets of ivermectin and other antiparasitic drugs. These ligand-gated Cl⁻ channels mediate inhibitory neurotransmission in nematodes and arthropods. Although GluCl channels are absent in mammals, their close relatives — **GABA-A receptors** (also Cl⁻ channels) and **glycine receptors** — are critical CNS safety targets. Off-target block or modulation of these channels can cause seizures, sedation, respiratory depression, or cognitive impairment. Just as hERG liability screening is mandatory for cardiac safety, screening against neuronal ion channels (GABA-A, NMDA, nAChR, Nav1.x, Kv channels) is essential for **CNS safety pharmacology** (ICH S7A). The computational and physiological evaluation principles are identical: rigorous controls, proper metrics, and honest assessment of model limitations.

### Electrophysiology: Shared Signal Processing from Fly Neurons to Human ECGs

Dr. Serbe-Kamp's electrophysiology experience — recording from identified neurons in the *Drosophila* optic lobe using patch-clamp and extracellular techniques — employs the **same fundamental signal processing principles** as clinical ECG analysis. In both cases, the raw signal is a voltage trace over time, contaminated by noise, and the goal is to extract meaningful features: peak amplitudes, intervals between events, frequency content, and waveform morphology. Bandpass filtering to isolate the signal of interest (e.g., 0.05–100 Hz for ECG, 300–3000 Hz for extracellular spikes), baseline correction to remove drift, and threshold-based event detection (R-peak detection in ECG, spike sorting in neural recordings) are shared techniques. The QT interval on an ECG is conceptually analogous to the **spike width** in a neural recording — both reflect the duration of ion channel–mediated repolarization, and both are prolonged when K⁺ channel function is compromised. This cross-disciplinary perspective reinforces a key theme of this course: the analytical tools you learn in one domain (neuroscience, cardiology, or machine learning) transfer powerfully to others.

In [ ]:
# ============================================================
# SIMULATED ECG AND QT INTERVAL MEASUREMENT
# This demonstrates how drug safety evaluation works:
# the QT interval on the ECG is the #1 cardiac safety biomarker.
# hERG-blocking drugs prolong QT -> risk of fatal arrhythmia.
# Reference: Redfern et al. (2003), Cardiovasc. Res. 58:32-45
# ============================================================

# Import numpy for numerical operations (arrays, math functions)
import numpy as np
# Import matplotlib for creating ECG waveform plots
import matplotlib.pyplot as plt

# Simulate a simplified ECG waveform
# A real ECG has P, QRS, T waves representing different phases of the cardiac cycle
# P wave: atrial depolarization (atria contract)
# QRS complex: ventricular depolarization (ventricles contract) - the big spike
# T wave: ventricular repolarization (ventricles recover)
# QT interval = time from Q wave to end of T wave = ventricular electrical cycle

def simulate_ecg_beat(t_beat, heart_rate=72, qt_prolonged=False):
    # t_beat: time array for one heartbeat
    # heart_rate: beats per minute (normal resting: 60-100 bpm)
    # qt_prolonged: if True, simulate hERG-blocking drug effect
    
    # Duration of one beat in seconds
    # 60 seconds/minute divided by beats/minute = seconds/beat
    beat_duration = 60.0 / heart_rate
    
    # Normalize time to [0, 1] for one beat cycle
    # This maps the absolute time to a fraction of the beat duration
    t_norm = (t_beat - t_beat[0]) / beat_duration
    
    # Create the ECG waveform as sum of Gaussian-like components
    # Each wave (P, QRS, T) is modeled as a Gaussian curve at the appropriate timing
    ecg = np.zeros_like(t_norm)
    
    # P wave: small positive deflection at ~0.15 of the cycle
    # Represents atrial depolarization (electrical activation of upper heart chambers)
    # Amplitude 0.15 mV, centered at 15% of the beat cycle, narrow width (sigma=0.01)
    ecg += 0.15 * np.exp(-((t_norm - 0.15)**2) / (2 * 0.01**2))
    
    # QRS complex: large spike at ~0.35 of the cycle
    # Q wave (small negative), R wave (large positive), S wave (small negative)
    # Represents ventricular depolarization - the main pumping event
    # Q wave: small downward deflection just before the R wave (amplitude -0.1 mV)
    ecg -= 0.1 * np.exp(-((t_norm - 0.33)**2) / (2 * 0.003**2))   # Q wave
    # R wave: the tallest peak on the ECG (amplitude 1.0 mV), represents the main ventricular depolarization
    ecg += 1.0 * np.exp(-((t_norm - 0.35)**2) / (2 * 0.005**2))   # R wave
    # S wave: small downward deflection just after the R wave (amplitude -0.15 mV)
    ecg -= 0.15 * np.exp(-((t_norm - 0.37)**2) / (2 * 0.003**2))  # S wave
    
    # T wave: positive deflection representing ventricular repolarization
    # This is what gets delayed by hERG-blocking drugs!
    if qt_prolonged:
        # Drug effect: T wave is delayed (appears later) and may be broader
        # This is what happens when a drug blocks the hERG potassium channel
        # The ventricles take longer to recover their electrical charge
        # Normal T wave at 0.55 is shifted to 0.65 = 10% of beat duration later
        t_wave_center = 0.65  # Delayed from normal 0.55
        # Broader T wave (increased sigma) is also a hallmark of hERG block
        t_wave_width = 0.025  # Broader T wave
    else:
        # Normal T wave timing: appears at ~55% of the beat cycle
        t_wave_center = 0.55  # Normal T wave timing
        # Normal T wave width (tighter Gaussian)
        t_wave_width = 0.02
    
    # Add the T wave component: amplitude 0.3 mV, Gaussian shape
    # The center and width depend on whether hERG is blocked (qt_prolonged flag)
    ecg += 0.3 * np.exp(-((t_norm - t_wave_center)**2) / (2 * t_wave_width**2))
    
    # Return the complete ECG waveform for one heartbeat
    return ecg

# Generate ECG traces: normal vs. QT-prolonged (drug effect)
# Sample at 500 Hz (standard clinical ECG sampling rate)
# Higher sampling rates capture the sharp QRS complex more accurately
fs = 500  # Sampling frequency in Hz
# Normal resting heart rate for a healthy adult
heart_rate = 72  # Normal resting heart rate
# Calculate the duration of one heartbeat in seconds (60/72 = 0.833 seconds)
beat_duration = 60.0 / heart_rate  # Duration of one beat in seconds
# Create a time array spanning 3 heartbeats, sampled at 500 Hz
# np.arange(start, stop, step) creates evenly spaced values
t = np.arange(0, 3 * beat_duration, 1/fs)  # 3 heartbeats

# Create normal ECG (no drug effect) - initialize with zeros
ecg_normal = np.zeros_like(t)
# Generate each of the 3 heartbeats separately and place them in the correct time window
for beat in range(3):
    # Calculate the start time of this beat
    start = beat * beat_duration
    # Calculate the end time of this beat
    end = (beat + 1) * beat_duration
    # Create a boolean mask selecting only the time points within this beat
    mask = (t >= start) & (t < end)
    # Generate the normal ECG waveform for this beat and place it in the output array
    ecg_normal[mask] = simulate_ecg_beat(t[mask], heart_rate, qt_prolonged=False)

# Create QT-prolonged ECG (simulating hERG-blocking drug like terfenadine or cisapride)
ecg_prolonged = np.zeros_like(t)
# Same loop structure, but with qt_prolonged=True to simulate drug effect
for beat in range(3):
    # Calculate the start time of this beat
    start = beat * beat_duration
    # Calculate the end time of this beat
    end = (beat + 1) * beat_duration
    # Create a boolean mask for this beat's time window
    mask = (t >= start) & (t < end)
    # Generate the QT-prolonged ECG waveform (T wave is delayed and broadened)
    ecg_prolonged[mask] = simulate_ecg_beat(t[mask], heart_rate, qt_prolonged=True)

# Plot comparison of normal vs. QT-prolonged ECG
# Create a figure with 2 vertically stacked subplots sharing the same x-axis (time)
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top subplot: Normal ECG waveform
# Plot time in milliseconds (multiply seconds by 1000) for clinical convention
# Blue line, 1.5pt width for clear visibility
axes[0].plot(t * 1000, ecg_normal, 'b-', linewidth=1.5)
# Label the y-axis with the ECG amplitude unit (millivolts)
axes[0].set_ylabel('Amplitude (mV)', fontsize=12)
# Title indicating this is a normal, healthy ECG
axes[0].set_title('Normal ECG - Healthy QT Interval', fontsize=14, fontweight='bold')
# Add a horizontal line at y=0 (the isoelectric baseline of the ECG)
axes[0].axhline(y=0, color='gray', linestyle='-', alpha=0.3)

# Bottom subplot: QT-prolonged ECG (drug effect)
# Red line to visually distinguish from the normal trace and indicate danger/abnormality
axes[1].plot(t * 1000, ecg_prolonged, 'r-', linewidth=1.5)
# Label the y-axis with amplitude units
axes[1].set_ylabel('Amplitude (mV)', fontsize=12)
# Label the x-axis with time units (only on bottom subplot since axes are shared)
axes[1].set_xlabel('Time (ms)', fontsize=12)
# Title indicating this is an abnormal ECG caused by hERG channel block
axes[1].set_title('QT-Prolonged ECG - hERG Channel Block (Drug Effect!)', fontsize=14, fontweight='bold')
# Add baseline reference line
axes[1].axhline(y=0, color='gray', linestyle='-', alpha=0.3)

# Adjust spacing between subplots to prevent label overlap
plt.tight_layout()
# Render and display the figure
plt.show()

# Print educational summary about ECG and drug safety
print("=== ECG & DRUG SAFETY ===")
# Display the heart rate used in the simulation
print(f"Heart rate: {heart_rate} bpm")
# Calculate and display the approximate normal QT interval
# 0.20 * beat_duration represents the normal QT fraction of a heartbeat
print(f"Normal QT interval: ~{0.20 * beat_duration * 1000:.0f} ms")
# Calculate and display the prolonged QT interval
# 0.30 * beat_duration represents the prolonged QT fraction
print(f"Prolonged QT interval: ~{0.30 * beat_duration * 1000:.0f} ms")
# Clinical danger threshold: QTc > 500 ms is associated with high TdP risk
print(f"\nQTc > 500 ms is DANGEROUS (risk of Torsades de Pointes)")
# List drugs withdrawn from market due to QT prolongation
print(f"\nDrugs withdrawn for QT prolongation:")
print(f"  - Terfenadine (antihistamine, withdrawn 1998)")
print(f"  - Cisapride (GI motility, withdrawn 2000)")
print(f"  - Astemizole (antihistamine, withdrawn 1999)")
# Connect back to the main theme: ML-based hERG prediction uses the same evaluation methods
print(f"\nhERG prediction is now a standard ML task in drug discovery!")
print(f"The SAME evaluation methods from this week (scaffold splits, SHAP)")
print(f"are used to validate hERG prediction models.")

## 6. Exercises

1. **Temporal split**: If the dataset had dates, how would you implement a temporal split?
2. **Different SHAP**: Use KernelSHAP instead of TreeSHAP. How do results compare?
3. **Classification**: Convert solubility to binary (soluble/insoluble at logS > -2). Compute ROC-AUC and PR-AUC.
4. **Challenge**: Implement conformal prediction intervals for your solubility model.

## References
- Bemis, G.W. & Murcko, M.A. (1996). The Properties of Known Drugs. J. Med. Chem. 39:2887-2893
- Lundberg, S.M. & Lee, S. (2017). A Unified Approach to Interpreting Model Predictions. NeurIPS
- Wallach, I. & Heifets, A. (2018). Most Ligand-Based Virtual Screening Benchmarks Reward Memorization. JCIM
- Sahigara, F. et al. (2012). Comparison of Different Approaches to Define the Applicability Domain. Molecules 17:4791-4810
- Redfern, W.S. et al. (2003). Relationships between preclinical cardiac electrophysiology, clinical QT interval prolongation and torsade de pointes for a broad range of drugs. Cardiovasc. Res. 58:32-45